## EDA And Feature Engineering Flight Price Prediction
check the dataset info below
https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction

### FEATURES
The various features of the cleaned dataset are explained below:
1) Airline: The name of the airline company is stored in the airline column. It is a categorical feature having 6 different airlines.
2) Flight: Flight stores information regarding the plane's flight code. It is a categorical feature.
3) Source City: City from which the flight takes off. It is a categorical feature having 6 unique cities.
4) Departure Time: This is a derived categorical feature obtained created by grouping time periods into bins. It stores information about the departure time and have 6 unique time labels.
5) Stops: A categorical feature with 3 distinct values that stores the number of stops between the source and destination cities.
6) Arrival Time: This is a derived categorical feature created by grouping time intervals into bins. It has six distinct time labels and keeps information about the arrival time.
7) Destination City: City where the flight will land. It is a categorical feature having 6 unique cities.
8) Class: A categorical feature that contains information on seat class; it has two distinct values: Business and Economy.
9) Duration: A continuous feature that displays the overall amount of time it takes to travel between cities in hours.
10)Days Left: This is a derived characteristic that is calculated by subtracting the trip date by the booking date.
11) Price: Target variable stores information of the ticket price.

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yahya4797/flight-price")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/yahya4797/flight-price


In [3]:
df = pd.read_excel("/kaggle/input/datasets/yahya4797/flight-price/flight_price (1).xlsx")

In [4]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 918.2+ KB


In [6]:
df.describe()

,Price
count,10683.000000
mean,9087.064121
std,4611.359167
min,1759.000000
25%,5277.000000
50%,8372.000000
75%,12373.000000
max,79512.000000


In [7]:
## Firstly converting data to datetime from object 
df['Date_of_Journey'].str.split('/')

0        [24, 03, 2019]
1         [1, 05, 2019]
2         [9, 06, 2019]
3        [12, 05, 2019]
4        [01, 03, 2019]
              ...      
10678     [9, 04, 2019]
10679    [27, 04, 2019]
10680    [27, 04, 2019]
10681    [01, 03, 2019]
10682     [9, 05, 2019]
Name: Date_of_Journey, Length: 10683, dtype: object

In [8]:
df['Date'] = df['Date_of_Journey'].str.split('/').str[0]
df['Month'] = df['Date_of_Journey'].str.split('/').str[1]
df['Year'] = df['Date_of_Journey'].str.split('/').str[2]

In [9]:
## our date , month , and year column are still object type so we need to convert 
## then into int 

df['Date'] = df['Date'].astype(int)
df['Month'] = df['Month'].astype(int)
df['Year'] = df['Year'].astype(int)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
 11  Date             10683 non-null  int64 
 12  Month            10683 non-null  int64 
 13  Year             10683 non-null  int64 
dtypes: int64(4), object(10)
memory usage: 1.1+ MB


In [11]:
## Drop date_of_journey column 
df.drop('Date_of_Journey' , axis = 1 , inplace = True )

In [12]:
## similar to the date we have to seperate the time , Firstly we need to just 
## get the time as in some of the data points date is also included 
df['Arrival_Time'] = df['Arrival_Time'].apply(lambda x:x.split(' ')[0])
df['Arrival_Hour'] = df['Arrival_Time'].str.split(':').str[0]
df['Arrival_Minute'] = df['Arrival_Time'].str.split(':').str[1]

In [13]:
df['Dep_Time'] = df['Dep_Time'].apply(lambda x : x.split(' ')[0])
df['Dep_Hour'] = df['Dep_Time'].str.split(':').str[0]
df['Dep_Minute'] = df['Dep_Time'].str.split(':').str[1]

In [14]:
df.drop('Dep_Time' , axis = 1 , inplace = True)
df.drop('Arrival_Time' , axis = 1 , inplace = True)

In [15]:
df['Dep_Hour'] = df['Dep_Hour'].astype(int)
df['Arrival_Hour'] = df['Arrival_Hour'].astype(int)
df['Dep_Minute'] = df['Dep_Minute'].astype(int)
df['Arrival_Minute'] = df['Arrival_Minute'].astype(int)

In [16]:
df.columns[df.isnull().sum() > 0]

Index(['Route', 'Total_Stops'], dtype='object')

In [17]:
df['Total_Stops'].unique()

array(['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops'],
      dtype=object)

In [18]:
## Giving numerical value to total_stops and purposely will be give it ordinal 
## values as price keeps on increasing as number of stop increases 
df['Total_Stops'] = df['Total_Stops'].map({
    'non-stop' : 0, 
    '1 stop' : 1 , 
    '2 stops' : 2 , 
    '3 stops' : 3,
    '4 stops' : 4 , 
    np.nan : 1 ## 1 is the mode of the data (1 stop)
    
})


In [19]:
df.Total_Stops.value_counts()

Total_Stops
1    5626
0    3491
2    1520
3      45
4       1
Name: count, dtype: int64

In [20]:
## Route column is not required 
df.drop('Route' , axis = 1 , inplace = True)

In [21]:
df.head()

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dep_Hour,Dep_Minute
0,IndiGo,Banglore,New Delhi,2h 50m,0,No info,3897,24,3,2019,1,10,22,20
1,Air India,Kolkata,Banglore,7h 25m,2,No info,7662,1,5,2019,13,15,5,50
2,Jet Airways,Delhi,Cochin,19h,2,No info,13882,9,6,2019,4,25,9,25
3,IndiGo,Kolkata,Banglore,5h 25m,1,No info,6218,12,5,2019,23,30,18,5
4,IndiGo,Banglore,New Delhi,4h 45m,1,No info,13302,1,3,2019,21,35,16,50


In [22]:
# Use the 'r' prefix for raw strings
# Extract numbers followed by 'h'
df['Duration_hour'] = df['Duration'].str.extract(r'(\d+)h')
# Extract numbers followed by 'm'
df['Duration_minute'] = df['Duration'].str.extract(r'(\d+)m')

In [23]:
## The na values in these newly constructed duration_hour and duration_minute
## means they had either 0 hours or 0 minutes 

df['Duration_hour'] = df['Duration_hour'].fillna(0)
df['Duration_minute'] = df['Duration_minute'].fillna(0)

In [24]:
df['Duration_hour'] = df['Duration_hour'].astype(int)
df['Duration_minute'] = df['Duration_minute'].astype(int)

In [25]:
df.drop('Duration' , axis = 1 , inplace = True)

In [26]:
df['Airline'].unique()

array(['IndiGo', 'Air India', 'Jet Airways', 'SpiceJet',
       'Multiple carriers', 'GoAir', 'Vistara', 'Air Asia',
       'Vistara Premium economy', 'Jet Airways Business',
       'Multiple carriers Premium economy', 'Trujet'], dtype=object)

In [27]:
df['Source'].unique()

array(['Banglore', 'Kolkata', 'Delhi', 'Chennai', 'Mumbai'], dtype=object)

In [28]:
df['Additional_Info'].unique()

array(['No info', 'In-flight meal not included',
       'No check-in baggage included', '1 Short layover', 'No Info',
       '1 Long layover', 'Change airports', 'Business class',
       'Red-eye flight', '2 Long layover'], dtype=object)

In [29]:
## in destination both new delhi and delhi are present which is same 
df['Destination'] = df['Destination'].replace('New Delhi' , 'Delhi')

In [30]:
### We can convert all these into numerical features using onehotencoding 
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
encoder.fit_transform(df[['Airline' , 'Source' , 'Additional_Info' , 'Destination']]).toarray()

array([[0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.]])

In [31]:
encoded_cols = pd.DataFrame(encoder.fit_transform(df[['Airline' , 'Source' , 'Additional_Info' , 'Destination']]).toarray() , 
            columns = encoder.get_feature_names_out())



In [32]:
df = pd.concat([df, encoded_cols], axis=1)

In [33]:
# Sirf list of names dena kafi hai
df.drop(['Airline', 'Source', 'Additional_Info' , 'Destination'], axis=1, inplace=True)

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 43 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Total_Stops                                   10683 non-null  int64  
 1   Price                                         10683 non-null  int64  
 2   Date                                          10683 non-null  int64  
 3   Month                                         10683 non-null  int64  
 4   Year                                          10683 non-null  int64  
 5   Arrival_Hour                                  10683 non-null  int64  
 6   Arrival_Minute                                10683 non-null  int64  
 7   Dep_Hour                                      10683 non-null  int64  
 8   Dep_Minute                                    10683 non-null  int64  
 9   Duration_hour                                 10683 non-null 